In [1]:
def Model_Reuse(dataset, window_size, target_column, windows_similarity_dictionary, forecasting_approach):
    saved_models = [] # store the models in a list
    for i in range(window_size, len(dataset), window_size): # loop through the dataset in windows
        # assign the train data and test data
        train = dataset[i - window_size:i]
        test = dataset[i: i + window_size] 
        y_train = train[target_column]
        X_train = train.drop(columns=[target_column])
        y_test = test[target_column]
        X_test = test.drop(columns=[target_column]) 

        if len(test) < window_size:  # break the "for" loop if the test size is less than window_size
            break
        # calculate window index for ES approach
        if forecasting_approach == "ES": 
            window_index = round(i / window_size) - 1
        elif forecasting_approach == "SA":
            window_index = round(i / window_size)

        # windows index and their corresponding similar window index is stored in the "windows_similarity_dictionary"
        
        if window_index in windows_similarity_dictionary: # check if window index is in similarity dictionary
            similar_window_index = windows_similarity_dictionary[window_index] #find similar window index
            if similar_window_index in windows_similarity_dictionary: # check if there is a chain in similarities
                
                # find the previous similarities in the chain and train a new model (M) on the test data of the previous element of the chain. Then use the model M on the recent test data
                if forecasting_approach == "ES": 
                    previous_model_i = (window_index+1) * window_size
                elif forecasting_approach == "SA":
                    previous_model_i = (window_index) * window_size
                test = dataset[previous_model_i: previous_model_i + window_size]
                y_test2 = test[target_column]
                X_test2 = test.drop(columns=[target_column]) 
                model = train_model(X_test2, y_test2)
                saved_models[similar_window_index] = model # replace the new trained model with the previous stored model
                saved_models.append(model) # save the new model
                y_pred = model.predict(X_test)
                mean_squared_error = mean_squared_error(y_test, y_pred) #calculate MSE
                mean_absolute_error = mean_absolute_error(y_test, y_pred) #calculate MAE
            else:
                model = saved_models[similar_window_index] #retrieve the existing model
                saved_models.append(model) # save the new model
                y_pred = model.predict(X_test)
                mean_squared_error = mean_squared_error(y_test, y_pred) #calculate MSE
                mean_absolute_error = mean_absolute_error(y_test, y_pred) #calculate MAE
        else:
            model= train_model(X_train, y_train)  # train a new model on the most recent window
            y_pred = model.predict(X_test)
            mean_squared_error = mean_squared_error(y_test, y_pred) #calculate MSE
            mean_absolute_error = mean_absolute_error(y_test, y_pred) #calculate MAE
            saved_models.append(model) # save the new model